# ECG Demo — Hybrid Verification

**Cell 2** — Custom `ecg_demo.bit`: sample ECG_DAC waveform via AXI MMIO  
**Cell 3** — Base overlay: drive DA4 via Pmod_IO + read AD2 via Pmod_IIC (loopback sweep)  
**Cell 4** — Plot both results

**Why two overlays?** The custom overlay drives DA4 via RTL SPI + reads AD2 via RTL I2C.  
When `base.bit` loads, the custom SPI driver is gone so DA4 outputs 0 V — we must re-drive it via Pmod_IO.

**Loopback wire required:** DA4 VOUT_A → AD2 CH0

In [ ]:
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

N_SAMPLES = 360
SLEEP_S   = 1.0 / 360
BASE_ADDR = 0x43C00000
REG_DAC   = 0x40

## Step 1 — Custom overlay: ECG_DAC waveform

In [ ]:
from pynq import Overlay, MMIO

print('Loading ecg_demo.bit ...')
ol = Overlay('/home/xilinx/pynq-ecg-demo/ps/ecg_demo.bit')
print('  Overlay loaded OK')

mmio = MMIO(BASE_ADDR, 0x44)

dac_vals = []
print(f'Sampling {N_SAMPLES} ECG_DAC points at ~360 Hz ...')
for _ in range(N_SAMPLES):
    dac_vals.append(mmio.read(REG_DAC) & 0xFFF)
    time.sleep(SLEEP_S)

dac_range = max(dac_vals) - min(dac_vals)
print(f'  ECG_DAC: min={min(dac_vals):#05x}  max={max(dac_vals):#05x}  range={dac_range}')
print('  DAC: ' + (f'PASS — DDS advancing ({dac_range} counts)' if dac_range > 10 else 'FAIL — DDS frozen'))

## Step 2 — Base overlay: loopback sweep (DA4 → AD2)

Base overlay replaces the custom bitstream, so DA4 is no longer driven by RTL.  
We drive it via `Pmod_IO` (bit-bang SPI) before reading the ADC — same method as `pmod_test.py`.

In [ ]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.pmod import Pmod_IO, Pmod_IIC, PMODA, PMODB

print('Loading base.bit (reprograms FPGA) ...')
base = BaseOverlay('base.bit')
print('  Base overlay loaded OK')

# ── DA4 SPI init (PMODA: CS=pin0/JA1, DIN=pin1/JA2, SCLK=pin3/JA4) ──
cs   = Pmod_IO(PMODA, 0, 'out')
mosi = Pmod_IO(PMODA, 1, 'out')
sclk = Pmod_IO(PMODA, 3, 'out')
cs.write(1); sclk.write(1)

def spi_write32(word):
    cs.write(0)
    for i in range(31, -1, -1):
        mosi.write((word >> i) & 1)
        sclk.write(0)
        sclk.write(1)
    cs.write(1)

def dac_set(ch, val):
    spi_write32((0x3 << 24) | ((ch & 0xF) << 20) | ((val & 0xFFF) << 8))

spi_write32(0x08000001)   # enable internal 2.5 V reference
time.sleep(0.01)
print('  DA4 internal 2.5 V reference enabled')

# ── AD2 I2C init (PMODB right half: SCL=pin2/JB3, SDA=pin3/JB4) ──
iic = Pmod_IIC(PMODB, 2, 3, 0x28)
print('  Pmod_IIC init OK (AD7991-0 @ 0x28)')

# ── Loopback sweep ──
SWEEP = [0, 512, 1024, 1536, 2048, 2560, 3072, 3584, 4095]
sweep_dac = []
sweep_adc = []

print('\n  Loopback sweep: DA4 VOUT_A → AD2 CH0')
print(f'  {"DAC code":>8}  {"DAC V":>6}  |  {"ADC raw":>7}  {"ADC V":>6}  {"Expected":>8}')
print('  ' + '-'*50)

monotonic = True
prev_adc  = None
for code in SWEEP:
    dac_set(0, code)
    time.sleep(0.1)
    iic.send([0x10])
    time.sleep(0.001)
    d = iic.receive(2)
    raw = ((d[0] & 0x0F) << 8) | d[1]
    sweep_dac.append(code)
    sweep_adc.append(raw)
    exp = int(code * 2.5 / 3.3)
    dac_v = code * 2.5 / 4095
    adc_v = raw  * 3.3 / 4096
    print(f'  {code:>8d}  {dac_v:>5.3f}V  |  {raw:>7d}  {adc_v:>5.3f}V  {exp:>8d}')
    if prev_adc is not None and code > 0 and raw < prev_adc - 50:
        monotonic = False
    prev_adc = raw

dac_set(0, 0)   # reset DA4 to 0 V

adc_range = max(sweep_adc) - min(sweep_adc)
print()
if monotonic and sweep_adc[-1] > sweep_adc[0] + 50:
    print(f'  ADC: PASS — tracks DA4 monotonically (range {adc_range} counts)')
else:
    print(f'  ADC: WARN — not monotonic or flat (range {adc_range}). Check loopback wire.')

## Step 3 — Plot

In [ ]:
t_ms = [i * (1000.0 / 360) for i in range(N_SAMPLES)]
dac_v_sweep = [c * 2.5 / 4095 for c in sweep_dac]
adc_v_sweep = [r * 3.3 / 4096 for r in sweep_adc]

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
fig.suptitle('ECG Demo — Hybrid Verification', fontsize=13)

axes[0].plot(t_ms, dac_vals, color='steelblue', linewidth=0.9)
axes[0].set_ylabel('Counts (0–4095)')
axes[0].set_title('ECG_DAC — DDS ROM output via AXI reg 0x40  (custom overlay)')
axes[0].set_xlabel('Time (ms)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(dac_v_sweep, adc_v_sweep, 'o-', color='darkorange', linewidth=1.2, markersize=6)
axes[1].plot([0, 2.5], [0, 2.5 * 4096 / 4095 * (1/3.3 * 3.3)], 'k--', linewidth=0.7, alpha=0.4, label='ideal (2.5 V Vref)')
axes[1].set_ylabel('ADC voltage (V)')
axes[1].set_xlabel('DA4 VOUT_A (V)')
axes[1].set_title('ADC CH0 vs DA4 VOUT_A — loopback sweep  (base overlay + Pmod_IIC)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
out = '/home/xilinx/jupyter_notebooks/hybrid_test.png'
plt.savefig(out, dpi=120)
plt.show()

print(f'Saved: {out}')
dac_ok = max(dac_vals) - min(dac_vals) > 10
adc_ok = max(sweep_adc) - min(sweep_adc) > 50
print(f'ECG_DAC : {"PASS" if dac_ok else "FAIL"}  (range {max(dac_vals)-min(dac_vals)} counts)')
print(f'ADC loop: {"PASS" if adc_ok else "WARN"}  (sweep range {max(sweep_adc)-min(sweep_adc)} counts)')
if dac_ok and adc_ok:
    print('\nOVERALL: PASS — DDS confirmed working, ADC hardware chain confirmed working')